# 04 -- Feature Build

> **Data mode: SYNTHETIC FIXTURE DATA.** Every filing/price/universe/sector record in this notebook comes from `_fixtures.py`, hand-authored for demonstration only. **Nothing computed in this notebook is a genuine historical result, and no cell here may be cited as evidence of reproducing `report_current.html`.** See `docs/reproducibility_findings.md` for the actual reproducibility status.

Runs Stage 3's real feature pipeline (`build_production_features`) over the synthetic bundle -- no formula in this notebook; every feature value comes from `atlas_quant.strategies.filing_momentum_ml.feature_pipeline`, unchanged.

In [ ]:
import atlas_quant
print("atlas_quant path:", atlas_quant.__file__)

import sys
sys.path.insert(0, ".")  # this notebook's own directory (for _fixtures.py) -- not a legacy path hack
import _fixtures


In [ ]:
from datetime import date, datetime
from atlas_quant.strategies.filing_momentum_ml.config import FeatureCacheIdentity, FilingMomentumMLConfig
from atlas_quant.strategies.filing_momentum_ml.production.feature_label_build import build_production_features
from atlas_quant.strategies.filing_momentum_ml.sector_encoding import SectorEncoder

bundle = _fixtures.build_synthetic_bundle()
calendar = _fixtures.build_trading_calendar(bundle)
periods = _fixtures.build_periods()
config = FilingMomentumMLConfig()
targets = [(i, p.quarter_end) for p in periods for i in bundle.universe]

cache_identity = FeatureCacheIdentity(
    strategy_id=config.strategy_id, strategy_version="notebook", feature_schema_version="1",
    fcf_mode=config.fcf_mode, train_years=config.ml_train_years, min_train_quarters=config.min_train_quarters,
    model_config_identity=config.identity(), universe_id="synthetic-notebook-universe",
    data_cutoff=date(2023, 6, 1), created_at=datetime(2023, 6, 1),
)
result = build_production_features(
    config=config, calendar=calendar, sector_encoder=SectorEncoder(), targets=targets,
    filings_by_instrument=bundle.filings_by_instrument, prices_by_instrument=bundle.prices_by_instrument,
    sector_by_instrument=bundle.sector_by_instrument,
    data_cutoff=max(p.evaluation_timestamp for p in periods),
    cache_identity=cache_identity, cache_root=None,  # never write to a real cache path from a notebook
)
print("blocked:", result.blocked)
for obs in result.feature_pipeline_result.observations:
    print(obs.instrument_id.symbol, obs.quarter_end, "missing_features:", obs.missing_features)
    print("  ", {k: round(v, 4) for k, v in obs.features.items()})


## Findings

- Both synthetic instruments produce a `FeatureObservation` for the target quarter, with a handful of missing features (the synthetic price series is too short/synthetic for every momentum/volatility window to be fully populated).

## Limitations

- These feature *values* are meaningless (derived from a smooth, synthetic price/revenue series) -- only the *mechanism* (point-in-time selection, timing resolution, missing-data handling) is being demonstrated.